In [1]:
import pandas as pd
import numpy as np

# =====================================================================
# STEP 1: DOWNLOAD AND FILTER RAW EPA DATA
# =====================================================================
print("1. Downloading EPA dataset...")
url = "https://www.fueleconomy.gov/feg/epadata/vehicles.csv.zip"
epa_df = pd.read_csv(url, low_memory=False)

# UPDATED: Added 'atvType' to differentiate PEVs and BEVs
cols_to_keep = ['year', 'make', 'model', 'VClass', 'fuelType1', 'atvType', 'comb08', 'range', 'co2TailpipeGpm']
df = epa_df[cols_to_keep].copy()

# =====================================================================
# STEP 2: MAP BODY AND FUEL TYPES
# =====================================================================
print("2. Mapping vehicle categories...")
body_mapping = {
    'Two Seaters': 'Car', 'Minicompact Cars': 'Car', 'Subcompact Cars': 'Car', 
    'Compact Cars': 'Car', 'Midsize Cars': 'Car', 'Large Cars': 'Car', 
    'Small Station Wagons': 'Car', 'Midsize Station Wagons': 'Car',
    'Sport Utility Vehicle - 2WD': 'SUV', 'Sport Utility Vehicle - 4WD': 'SUV',
    'Small Sport Utility Vehicle 2WD': 'SUV', 'Small Sport Utility Vehicle 4WD': 'SUV',
    'Standard Sport Utility Vehicle 2WD': 'SUV', 'Standard Sport Utility Vehicle 4WD': 'SUV',
    'Minivan - 2WD': 'Van', 'Minivan - 4WD': 'Van', 'Vans': 'Van', 
    'Vans, Cargo Type': 'Van', 'Vans, Passenger Type': 'Van',
    'Standard Pickup Trucks': 'Pickup', 'Standard Pickup Trucks 2WD': 'Pickup', 
    'Standard Pickup Trucks 4WD': 'Pickup', 'Small Pickup Trucks': 'Pickup', 
    'Small Pickup Trucks 2WD': 'Pickup', 'Small Pickup Trucks 4WD': 'Pickup',
    'Motorcycles': 'Motorcycle'
}

df['body_type'] = df['VClass'].map(body_mapping)

# UPDATED: Advanced fuel mapping using both atvType and fuelType1
def map_fuel(row):
    atv = str(row['atvType'])
    fuel1 = str(row['fuelType1'])
    
    if 'Plug-in' in atv: return 'PEV'
    if 'EV' in atv or 'Electricity' in fuel1: return 'BEV'
    if 'Hybrid' in atv: return 'Hybrid'
    if 'Diesel' in fuel1: return 'Diesel'
    if 'Gasoline' in fuel1: return 'Gas'
    return 'Other'

df['fuel_type'] = df.apply(map_fuel, axis=1)
df = df.dropna(subset=['body_type'])
df = df[df['fuel_type'] != 'Other']

# =====================================================================
# STEP 3: AGGREGATE REAL DATA
# =====================================================================
print("3. Aggregating historical data...")
grouped = df.groupby(['year', 'body_type', 'fuel_type'])
agg_df = grouped.agg(
    NumMakes=('make', 'nunique'),
    NumModels=('model', 'nunique'),
    MPG=('comb08', 'mean'),
    Range=('range', 'mean'),
    co2gpm=('co2TailpipeGpm', 'mean')
).reset_index()

# =====================================================================
# STEP 4: RECTANGULAR MATRIX & SMART IMPUTATION (2007-2026)
# =====================================================================
print("4. Building rectangular matrix and imputing missing technology...")
years = list(range(2007, 2027))
bodies = ['Car', 'SUV', 'Van', 'Pickup', 'Motorcycle']
fuels = ['Gas', 'Diesel', 'Hybrid', 'BEV', 'PEV']

idx = pd.MultiIndex.from_product([years, bodies, fuels], names=['year', 'body_type', 'fuel_type'])
full_universe = pd.DataFrame(index=idx).reset_index()

merged_df = pd.merge(full_universe, agg_df, on=['year', 'body_type', 'fuel_type'], how='left')

# Zero out availability for nonexistent cars
merged_df['NumMakes'] = merged_df['NumMakes'].fillna(0).astype(int)
merged_df['NumModels'] = merged_df['NumModels'].fillna(0).astype(int)

# Extract real data to calculate Tech Ratios vs. Gas
real_data = merged_df.dropna(subset=['MPG'])
gas_baseline = real_data[real_data['fuel_type'] == 'Gas'].set_index(['year', 'body_type'])[['MPG', 'Range', 'co2gpm']]

ratios = {}
for fuel in fuels:
    if fuel == 'Gas': continue
    fuel_data = real_data[real_data['fuel_type'] == fuel].set_index(['year', 'body_type'])
    comparison = fuel_data.join(gas_baseline, lsuffix='_fuel', rsuffix='_gas').dropna()
    
    if not comparison.empty:
        ratios[fuel] = {
            'MPG': (comparison['MPG_fuel'] / comparison['MPG_gas']).mean(),
            'Range': (comparison['Range_fuel'] / comparison['Range_gas']).mean(),
            'co2gpm': (comparison['co2gpm_fuel'] / comparison['co2gpm_gas']).mean()
        }
    else:
        ratios[fuel] = {'MPG': 1.0, 'Range': 1.0, 'co2gpm': 1.0}

# Apply Ratios to impute missing data
merged_df = merged_df.merge(gas_baseline.reset_index(), on=['year', 'body_type'], suffixes=('', '_gas_base'), how='left')

for fuel in fuels:
    if fuel == 'Gas': continue
    mask = (merged_df['fuel_type'] == fuel) & (merged_df['MPG'].isna())
    merged_df.loc[mask, 'MPG'] = merged_df.loc[mask, 'MPG_gas_base'] * ratios[fuel]['MPG']
    merged_df.loc[mask, 'Range'] = merged_df.loc[mask, 'Range_gas_base'] * ratios[fuel]['Range']
    merged_df.loc[mask, 'co2gpm'] = merged_df.loc[mask, 'co2gpm_gas_base'] * ratios[fuel]['co2gpm']

merged_df = merged_df.drop(columns=['MPG_gas_base', 'Range_gas_base', 'co2gpm_gas_base'])
merged_df = merged_df.fillna(0)

# =====================================================================
# STEP 5: ESTIMATE PRICING AND OPERATING COSTS
# =====================================================================
print("5. Estimating pricing and operating costs...")
def estimate_price_and_cost(row):
    base_prices = {'Car': 35000, 'SUV': 45000, 'Van': 40000, 'Pickup': 50000, 'Motorcycle': 15000}
    fuel_multiplier = {'Gas': 1.0, 'Diesel': 1.1, 'Hybrid': 1.15, 'PEV': 1.25, 'BEV': 1.4}
    
    base_price = base_prices[row['body_type']] * fuel_multiplier[row['fuel_type']]
    age = 2026 - row['year']
    depreciated_price = base_price * ((0.95) ** age)
    
    mpg = row['MPG'] if row['MPG'] > 0 else 25 
    if row['fuel_type'] == 'BEV':
        op_cost = ((33.7 / mpg) * 0.15 * 100) + 6 
    else:
        op_cost = ((3.50 / mpg) * 100) + 10 
        
    return pd.Series([round(depreciated_price, 2), round(op_cost, 2)])

merged_df[['NewPrice', 'auto_operating_cost']] = merged_df.apply(estimate_price_and_cost, axis=1)

# =====================================================================
# STEP 6: FORMAT AND EXPORT
# =====================================================================
print("6. Formatting and exporting to CSV...")
cols_order = [
    'body_type', 'fuel_type', 'year', 'NumMakes', 'NumModels', 
    'MPG', 'Range', 'NewPrice', 'auto_operating_cost', 'co2gpm'
]

final_df = merged_df.sort_values(by=['body_type', 'fuel_type', 'year'], ascending=[True, True, False])
final_df = final_df[cols_order]

# FIX: Find any Infinities (from dividing by zero) and turn them to 0
final_df = final_df.replace([np.inf, -np.inf], np.nan).fillna(0)

# Final rounding and type enforcement
final_df['NumMakes'] = final_df['NumMakes'].astype(int)
final_df['NumModels'] = final_df['NumModels'].astype(int)
final_df['Range'] = final_df['Range'].astype(int)
final_df['MPG'] = final_df['MPG'].round(1)
final_df['co2gpm'] = final_df['co2gpm'].round(2)

# Set the exact index ActivitySim needs
final_df['age'] = 2026 - final_df['year']
final_df['vehicle_type'] = final_df['body_type'] + '_' + final_df['age'].astype(str) + '_' + final_df['fuel_type']
final_df.set_index('vehicle_type', inplace=True)
final_df = final_df.drop(columns=['age'])

final_df.to_csv('vehicle_type_data_2026.csv')
print(f"\nDone! Exported {len(final_df)} rows to 'vehicle_type_data_2026.csv'.")

1. Downloading EPA dataset...
2. Mapping vehicle categories...
3. Aggregating historical data...
4. Building rectangular matrix and imputing missing technology...
5. Estimating pricing and operating costs...
6. Formatting and exporting to CSV...

Done! Exported 500 rows to 'vehicle_type_data_2026.csv'.


In [2]:
final_df`

,body_type,fuel_type,year,NumMakes,NumModels,MPG,Range,NewPrice,auto_operating_cost,co2gpm
vehicle_type,,,,,,,,,,
Car_0_BEV,Car,BEV,2026,13,89,95.9,298,49000.00,11.27,0.00
Car_1_BEV,Car,BEV,2025,16,103,95.2,291,46550.00,11.31,0.00
Car_2_BEV,Car,BEV,2024,16,96,98.0,284,44222.50,11.16,0.00
Car_3_BEV,Car,BEV,2023,14,68,102.4,279,42011.37,10.94,0.00
Car_4_BEV,Car,BEV,2022,13,49,99.8,280,39910.81,11.07,0.00
...,...,...,...,...,...,...,...,...,...,...
Van_15_PEV,Van,PEV,2011,0,0,20.0,0,23164.56,27.46,243.97
Van_16_PEV,Van,PEV,2010,0,0,22.8,0,22006.33,25.36,211.89
Van_17_PEV,Van,PEV,2009,0,0,21.8,0,20906.02,26.07,221.03


In [4]:
final_df.year.max()

np.int64(2026)